In [53]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# 01. Initial Data Inspection + Data understanding

Первым делом я собираюсь ознакомиться с данными, просто просмотреть их, понять вообще с чем работаем. В этой секции просто выведем данные на экран, посмотрим что есть за поля, возможно базовые дефекты данных которые будут видны без детального анализа.

## Item Master 

In [5]:
item_master_df = pd.read_csv("../data/item_master.csv")
item_master_df.head(), item_master_df.tail()

(        sku   category abc_class launch_period predecessor_sku  uom
 0  SKU_0001      SNACK         B       2023-01             NaN  PCS
 1  SKU_0002      DAIRY         C       2023-01             NaN  PCS
 2  SKU_0003  HOUSEHOLD         A       2023-01             NaN  PCS
 3  SKU_0004   BEVERAGE         B       2023-01             NaN  PCS
 4  SKU_0005      SNACK         C       2023-01             NaN  PCS,
          sku   category abc_class launch_period predecessor_sku  uom
 35  SKU_0036   BEVERAGE         A       2024-10        SKU_0011  PCS
 36  SKU_0037      SNACK         B       2023-01             NaN  PCS
 37  SKU_0038      DAIRY         C       2023-01             NaN  PCS
 38  SKU_0039  HOUSEHOLD         A       2023-01             NaN  PCS
 39  SKU_0040   BEVERAGE         B       2023-01             NaN  PCS)

Первый файл - **item_master.csv**. Поля и как я их понимаю:
1. **sku** - stock keeping unit (погуглил расшифровку). Идентификатор товара, далее могу упоиминать это поле как айди товара, в целом оно это собой и подразумевает.
2. **category** - категория, или класс товара. Подразумевает собой некоторое групповое обобщение товаров
    - todo: проверим какие есть категории, посмотрим как часто каждая из них встречается, мб будет полезно в будущем
3. **abc_class** - как я понял это результаты ABC-анализа (анализ важности товаров где класс A - самые важные товары, класс B - средние по важности товары, класс C - товары формирующее тяжелый хвост). Тут наверное стоит руководствоваться правилом Парето (80/20).
    - note: согласно Парето 20% класса A формируют 80 % продаж. Это будет супер важным для таски, т.к. ошибка прогнозирования для товаров класса A будет очень дорогой. Будет одним из ключевых факторов выбора модели прогнозирования.
    - note: *Если для нас ошибка на классе A будет критичной, но ошибкой на классе C мы можем более менее пренебречь, стоит подумать о каком-нибудь кастомном асимитричном лоссе. Будем штрафовать модель больше за ошибку на классе A и меньше за ошибку на классе C*.
4. launch_period - дата старта товара. Отсюда поймем откуда стоит ожидать начало ряда для товара и выделить новинки.
5. predecessor_sku - товар предшественник я полангаю, что-то типа обновленного товара, по факту явлющимся все тем же товаром или его продолжением.
    - note: поскольку это товар один и тот же надо будет смержить предшественника и текущего, чтобы получить истинный ряд.
6. uom - или unit of measure, в каких единицах товар измеряется. Возиожно не все товары будут измеряться в штуках и надо будет что-то делать чтобы привести к олдному скейлу.

In [6]:
item_master_df.__len__(), item_master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   sku              40 non-null     str  
 1   category         40 non-null     str  
 2   abc_class        40 non-null     str  
 3   launch_period    40 non-null     str  
 4   predecessor_sku  1 non-null      str  
 5   uom              40 non-null     str  
dtypes: str(6)
memory usage: 2.0 KB


(40, None)

Всего 40 строк, все поля кроме predecessor_sku не имеют пропусков что уже радует.
predecessor_sku - только 1 ненулевое значение. Вряд ли это дефект данных, просто единственный товар у которого есть предшественник.

In [7]:
item_master_df["category"].value_counts()

category
SNACK        10
DAIRY        10
HOUSEHOLD    10
BEVERAGE     10
Name: count, dtype: int64

In [8]:
item_master_df["abc_class"].value_counts()

abc_class
B    14
C    13
A    13
Name: count, dtype: int64

In [9]:
item_master_df["uom"].value_counts()

uom
PCS    40
Name: count, dtype: int64

- **category** - 4 возможных значения ["SNACK", "DAIRY", "HOUSEHOLD", "BEVERAGE"], по 10 штук каждого
- **abc_class** - 13 A, 14 B, 13 C
- **uom** - все товары считаются в штуках

## Promo Calendar

Календарь проведния скидок. Будет полезным знать когда на товары действовали скидки. Зачастую в такие периоды замечается больший спрос на товары, соответсвтенно требуется большая закупка. В такие периоды для нас будет очень критичным предсказать меньше чем на самом деле. 

In [10]:
promo_calendar_df = pd.read_csv("../data/promo_calendar.csv")
promo_calendar_df.head()

,sku,period,promo_type,discount_pct
0,SKU_0001,2023-11,TPR,25
1,SKU_0001,2024-11,TPR,25
2,SKU_0006,2024-03,TPR,25
3,SKU_0013,2025-04,TPR,25
4,SKU_0020,2023-06,TPR,25


In [11]:
promo_calendar_df.__len__()

8

1. **sku** - айди товара
2. **period** - в какой период проводилась акция, возможно будет полезным бинарным признаком типа is_dicount [True/False]
3. **promo_type** - вид акции (тут типа скидка, 2+1 и т.п. я полагаю)
4. **discount_pct** - скидка в процентах, мб кстати тоже будет полезным признаком, можно будет придумать что-то типа вместо реального прайса, прайс за который купили/купят как дополнительная фича

In [12]:
promo_calendar_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   sku           8 non-null      str  
 1   period        8 non-null      str  
 2   promo_type    8 non-null      str  
 3   discount_pct  8 non-null      int64
dtypes: int64(1), str(3)
memory usage: 388.0 bytes


In [13]:
promo_calendar_df["promo_type"].value_counts()

promo_type
TPR    8
Name: count, dtype: int64

У нас единственная акция - скидка по цене

In [14]:
promo_calendar_df["discount_pct"].value_counts()

discount_pct
25    8
Name: count, dtype: int64

Все товары по скидке продавались со скидкой 25 %

## Sales History

Истороия продажи товаров, отсюда мы и будем делать временные ряды.
Здесь будет важным правильно сгруппировать и аггрешировать данные.


**ВАЖНО!!!** Учесть воозможность пропущенных дат и подумать как их заполнить в случае пропусков.

In [15]:
sales_history_df = pd.read_csv("../data/sales_history.csv")
sales_history_df.head()

,sku,location,period,qty
0,SKU_0001,MSK,2023-01,243
1,SKU_0001,MSK,2023-02,179
2,SKU_0001,MSK,2023-03,168
3,SKU_0001,MSK,2023-04,204
4,SKU_0001,MSK,2023-05,153


1. **sku** - айди товара, будем джоинить по нему.
2. **location** - территориальная локация, в каком месте продавался товар. Стоит понимать, что в разных местах товар будет продаваться по-разному, в более населённых пунктах товар может покупаться чаще в целом, при эотм в среднем на душу населения может выходить одинаково. Было бы неплохо иметь маппинг с количеством населения, чтобы сделать фичу такого рода.
3. period - дата (в формате год-месяц), будет формировать хронология ряда
4. qty - сколько было продано товаров, с этим будет всё просто - к счастью все товары продавались поштучно

In [16]:
sales_history_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1526 entries, 0 to 1525
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   sku       1526 non-null   str  
 1   location  1526 non-null   str  
 2   period    1526 non-null   str  
 3   qty       1526 non-null   int64
dtypes: int64(1), str(3)
memory usage: 47.8 KB


Нет пропусков - гуд

### Чек по количеству проданных товаров в разных локациях

In [17]:
sold_items_location_wise_grouped = sales_history_df.groupby(by=["sku" ,"location"]).agg({
    "qty": np.sum,
    "period": len
}).reset_index().rename(columns={"qty": "total_count", "period": "total_months"})
sold_items_location_wise_grouped["mean_count_per_month"] = sold_items_location_wise_grouped.apply(
    lambda x: x["total_count"] / x["total_months"],
    axis=1
    )
sold_items_location_wise_grouped

,sku,location,total_count,total_months,mean_count_per_month
0,SKU_0001,MSK,12115,36,336.527778
1,SKU_0002,MSK,311,36,8.638889
2,SKU_0003,EKB,9791,36,271.972222
3,SKU_0003,MSK,22775,36,632.638889
4,SKU_0004,MSK,25824,37,697.945946
5,SKU_0005,MSK,20279,32,633.718750
6,SKU_0006,MSK,4816,36,133.777778
7,SKU_0007,MSK,51071,36,1418.638889
8,SKU_0008,EKB,8724,36,242.333333
9,SKU_0008,MSK,26094,36,724.833333


In [18]:
items_sold_in_msk_and_ekb = sold_items_location_wise_grouped[sold_items_location_wise_grouped.duplicated(subset="sku", keep=False)].reset_index(drop=True)
items_sold_in_msk_and_ekb

,sku,location,total_count,total_months,mean_count_per_month
0,SKU_0003,EKB,9791,36,271.972222
1,SKU_0003,MSK,22775,36,632.638889
2,SKU_0008,EKB,8724,36,242.333333
3,SKU_0008,MSK,26094,36,724.833333
4,SKU_0017,EKB,1146,36,31.833333
5,SKU_0017,MSK,4419,36,122.750000
6,SKU_0022,EKB,24812,36,689.222222
7,SKU_0022,MSK,58802,36,1633.388889
8,SKU_0027,EKB,18412,36,511.444444
9,SKU_0027,MSK,52900,36,1469.444444


In [19]:
items_sold_in_msk_and_ekb.groupby(by="location").agg({
    "mean_count_per_month": np.mean
}).reset_index()

,location,mean_count_per_month
0,EKB,347.069444
1,MSK,878.416667


В среднем товары, которые продаются и в Москве и в ЕКБ, в Москве продаются чуть больше чем в два раза чаще

### Статистика по товарам в МСК и ЕКБ

In [20]:
msc_only_df = sales_history_df[sales_history_df["location"] == "MSK"]
msc_only_df

,sku,location,period,qty
0,SKU_0001,MSK,2023-01,243
1,SKU_0001,MSK,2023-02,179
2,SKU_0001,MSK,2023-03,168
3,SKU_0001,MSK,2023-04,204
4,SKU_0001,MSK,2023-05,153
...,...,...,...,...
1521,SKU_0040,MSK,2025-08,0
1522,SKU_0040,MSK,2025-09,27
1523,SKU_0040,MSK,2025-10,0
1524,SKU_0040,MSK,2025-11,21


In [21]:
msc_only_items_qty_report = msc_only_df.groupby(by="sku").describe()["qty"]
msc_only_items_qty_statistics = pd.DataFrame(
    {
        "Mean Sold Items": [np.mean(msc_only_items_qty_report["mean"])],
        "Min Sold Items": [np.min(msc_only_items_qty_report["min"])],
        "Max Sold Items": [np.max(msc_only_items_qty_report["max"])],
    }
)
msc_only_items_qty_statistics

,Mean Sold Items,Min Sold Items,Max Sold Items
0,4874.408158,-8.0,406748.0


- В среднем в Москве продавалось 4874 различных товара в день
- Минимум проданных товаров = -8, отрицательное значение может быть:
    - Некорректность данных
    - Возвраты
- Максимум в Москве было продано 406748 товаров за день

In [22]:
ekb_only_df = sales_history_df[sales_history_df["location"] == "EKB"]
ekb_only_df

,sku,location,period,qty
72,SKU_0003,EKB,2023-01,358
73,SKU_0003,EKB,2023-02,313
74,SKU_0003,EKB,2023-03,286
75,SKU_0003,EKB,2023-04,216
76,SKU_0003,EKB,2023-05,192
...,...,...,...,...
1321,SKU_0034,EKB,2025-08,342
1322,SKU_0034,EKB,2025-09,339
1323,SKU_0034,EKB,2025-10,461
1324,SKU_0034,EKB,2025-11,471


In [23]:
ekb_only_items_qty_report = ekb_only_df.groupby(by="sku").describe()["qty"]
ekb_only_items_qty_statistics = pd.DataFrame(
    {
        "Mean Sold Items": [np.mean(ekb_only_items_qty_report["mean"])],
        "Min Sold Items": [np.min(ekb_only_items_qty_report["min"])],
        "Max Sold Items": [np.max(ekb_only_items_qty_report["max"])],
    }
)
ekb_only_items_qty_statistics

,Mean Sold Items,Min Sold Items,Max Sold Items
0,347.069444,20.0,1060.0


- В ЕКБ в среднем продавалось 347 товаров в день
- В ЕКБ минимум 20 товаров продавлаось в день, нет возвратов что довольно странно, возможно отрицательное значение и правда может быть ошибкой данных
- В ЕКБ максимум было продано 1060 товаров за день

### Отрицательные продажи

In [24]:
negative_sales = sales_history_df[sales_history_df["qty"] < 0]
negative_sales

,sku,location,period,qty
539,SKU_0014,MSK,2024-06,-8


Существует только 1 товар с отрицательным количеством продаж. Поскольку он всего один - будем полагать, что это ошибка данных и рассценивать отрицательные продажи как 0

### Чек продаж по месяцам + Базовая проверка сезонности

In [25]:
from enum import Enum


class MonthEnum(Enum):
    JAN = 1
    FEB = 2
    MAR = 3
    APR = 4
    MAY = 5
    JUN = 6
    JUL = 7
    AUG = 8
    SEP = 9
    OCT = 10
    NOV = 11
    DEC = 12

In [26]:
sales_history_df["period"] = pd.to_datetime(sales_history_df["period"])
sales_history_df["month"] = pd.DatetimeIndex(sales_history_df["period"]).month
sales_history_df["month"] = sales_history_df["month"].apply(lambda x: MonthEnum(x).name)
sales_history_df

,sku,location,period,qty,month
0,SKU_0001,MSK,2023-01-01,243,JAN
1,SKU_0001,MSK,2023-02-01,179,FEB
2,SKU_0001,MSK,2023-03-01,168,MAR
3,SKU_0001,MSK,2023-04-01,204,APR
4,SKU_0001,MSK,2023-05-01,153,MAY
...,...,...,...,...,...
1521,SKU_0040,MSK,2025-08-01,0,AUG
1522,SKU_0040,MSK,2025-09-01,27,SEP
1523,SKU_0040,MSK,2025-10-01,0,OCT
1524,SKU_0040,MSK,2025-11-01,21,NOV


In [27]:
sales_group_by_month = sales_history_df.groupby(by="month").agg(
    {
        "qty": np.sum,
    }
).reset_index().rename(columns={"qty": "qty_per_month"})
sales_group_by_month.sort_values(by="qty_per_month", ascending=False).reset_index(drop=True)

,month,qty_per_month
0,JUL,892346
1,SEP,876078
2,AUG,783666
3,OCT,770388
4,NOV,758056
5,DEC,649908
6,JUN,464943
7,MAY,425643
8,APR,420745
9,MAR,349780


Топ продаж приходится на Июль, Сентябрь и Август. Преимущественно продажи высоки летом и осенью. \
Продажи достаточно низки в начале года (Январь - Март)

In [28]:
sales_history_df_group_by_year_and_month = sales_history_df.groupby(by="period").agg({
    "qty": np.sum,
    "month": "first"
}).reset_index()
sales_history_df_group_by_year_and_month

,period,qty,month
0,2023-01-01,31839,JAN
1,2023-02-01,31755,FEB
2,2023-03-01,32349,MAR
3,2023-04-01,32447,APR
4,2023-05-01,31190,MAY
5,2023-06-01,44186,JUN
6,2023-07-01,29271,JUL
7,2023-08-01,34147,AUG
8,2023-09-01,31758,SEP
9,2023-10-01,32942,OCT


В целом кажется, что сезонности нет, продажи стабильны в течение года. \
Что-то произошло в Июле 2024 года, из-за чего продажи выросли в 10 раз, однако после этого они остались так же более менее стабильны стабильны.
Можем еще проверить на автокорелляционной функции.

In [55]:
from statsmodels.tsa.stattools import adfuller

df_grouped = sales_history_df.groupby(['sku', 'period'])['qty'].sum().reset_index()
ts_df = df_grouped.pivot(index='period', columns='sku', values='qty')
ts_df = ts_df.asfreq('MS')

p_values = {
    col: adfuller(ts_df[col].dropna())[1] 
    for col in ts_df.columns 
    if len(ts_df[col].dropna()) > 3
}

stationarity_report = pd.DataFrame.from_dict(
    p_values, 
    orient='index', 
    columns=['p-value']
)
stationarity_report.index.name = 'sku'

stationarity_report["is_stationary"] = stationarity_report["p-value"] <= 0.05
stationarity_report

,p-value,is_stationary
sku,,
SKU_0001,2.336121e-05,True
SKU_0002,1.822552e-05,True
SKU_0003,2.252909e-04,True
SKU_0004,3.707740e-04,True
SKU_0005,4.422986e-03,True
SKU_0006,9.936152e-06,True
SKU_0007,2.902968e-02,True
SKU_0008,4.794697e-02,True
SKU_0009,1.001234e-05,True


In [57]:
non_stationary = stationarity_report[~stationarity_report["is_stationary"]]
non_stationary

,p-value,is_stationary
sku,,
SKU_0011,0.136285,False
SKU_0018,0.571193,False
SKU_0022,0.981771,False
SKU_0023,0.999080,False
SKU_0026,0.789686,False
SKU_0029,0.930744,False
SKU_0030,0.232649,False
SKU_0033,0.089736,False
SKU_0035,0.607960,False


10 товаров с нестационарным рядом, означает что имеется тренд/сезонность. Возможно будет ещё одним параметром при выборе модели.

### Чек дубликатов по sku, period, location

In [36]:
sales_history_df[sales_history_df.duplicated(subset=["sku", "period", "location"], keep=False)]

,sku,location,period,qty,month
156,SKU_0004,MSK,2024-01-01,609,JAN
157,SKU_0004,MSK,2024-01-01,609,JAN
763,SKU_0019,MSK,2025-02-01,104,FEB
764,SKU_0019,MSK,2025-02-01,104,FEB
1148,SKU_0030,MSK,2023-04-01,571,APR
1149,SKU_0030,MSK,2023-04-01,571,APR


Есть дублирующиеся строки, из-за чего выше приведённая статистика может быть слегка неверна.

## Stock History

In [33]:
stock_history_df = pd.read_csv("../data/stock_history.csv")
stock_history_df.head()

,sku,location,period,stock_end_qty,days_out_of_stock
0,SKU_0001,MSK,2023-01,174,0
1,SKU_0001,MSK,2023-02,150,0
2,SKU_0001,MSK,2023-03,193,0
3,SKU_0001,MSK,2023-04,232,0
4,SKU_0001,MSK,2023-05,199,0
